In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
nunenuh_pytorch_challange_flower_dataset_path = kagglehub.dataset_download('nunenuh/pytorch-challange-flower-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    break  # faqat birinchi papkani ko'rsatadi

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input


In [ ]:
import os
import pandas as pd

data_dir = '/kaggle/input/datasets/nunenuh/pytorch-challange-flower-dataset/dataset' # dataset joylashgan manzil

# Barcha rasm yo'llarini va labellarini yig'ish
rows = []
for split in ['train', 'valid', 'test']: # 3 ta papkani birma-bir ko'rib chiqadi.
    split_dir = os.path.join(data_dir, split)
    for label in os.listdir(split_dir): # Har bir gul papkasini (1, 2, 3... 102) ko'rib chiqadi.
        label_dir = os.path.join(split_dir, label)
        if os.path.isdir(label_dir):
            for img in os.listdir(label_dir): # Har bir rasmni topib, yo'li + labeli + spliti bilan ro'yxatga qo'shadi.
                rows.append({
                    'path': os.path.join(label_dir, img),
                    'label': label,
                    'split': split
                })

df = pd.DataFrame(rows)
print(df.shape)
print(df.head())
print(df['split'].value_counts())

(7370, 3)
                                                path label  split
0  /kaggle/input/datasets/nunenuh/pytorch-challan...     7  train
1  /kaggle/input/datasets/nunenuh/pytorch-challan...     7  train
2  /kaggle/input/datasets/nunenuh/pytorch-challan...     7  train
3  /kaggle/input/datasets/nunenuh/pytorch-challan...     7  train
4  /kaggle/input/datasets/nunenuh/pytorch-challan...     7  train
split
train    6552
valid     818
Name: count, dtype: int64


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class FlowerDataset(Dataset): # Bu torch.utils.data.Dataset klassidan meros oladi. PyTorch Dataseti bo'lishi uchun shu klassdan voris olish shart.
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
# self.classes va self.class_to_idx: Model gul nomlarini (masalan, "1", "2") tushunmaydi, unga sonlar (0, 1, 2...) kerak. Bu qatorlar matnli nomlarni indekslarga o'girib beradigan lug'at yaratadi.

    def __len__(self): # Model datasetda nechta rasm borligini bilishi uchun jadvalimizdagi qatorlar sonini qaytaradi.
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx] # Berilgan indeksdagi qatorni oladi.
        image = Image.open(row['path']).convert('RGB')
# Image.open(...): Rasm faylini diskdan ochadi. .convert('RGB') esa rasmlar qora-oq yoki shaffof bo'lib qolmasligi
# uchun ularni standart 3 kanalli (Qizil, Yashil, Ko'k) rangga o'tkazadi.
        label = self.class_to_idx[row['label']]
# label = ...: "102" degan nomni class_to_idx lug'ati orqali mos raqamga (masalan, 101 ga) o'tkazadi.
        if self.transform:
            image = self.transform(image)
        return image, label # Tayyor rasm (tensor) va uning raqamli klassini qaytaradi.

print("✅ FlowerDataset klassi tayyor!")

✅ FlowerDataset klassi tayyor!


In [ ]:
df


,path,label,split
0,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train
1,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train
2,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train
3,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train
4,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train
...,...,...,...
7365,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid
7366,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid
7367,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid
7368,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid


In [ ]:
df.shape

(7370, 3)

In [ ]:
test_df = df[df['split'] == 'test']
print(test_df.shape)

(0, 3)


In [ ]:
print("Unique labellar soni:", df['label'].nunique())
print("\nBarcha labellar:")
print(sorted(df['label'].unique()))

Unique labellar soni: 102

Barcha labellar:
['1', '10', '100', '101', '102', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '6', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '7', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '8', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '9', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99']


In [ ]:
import json

with open('/kaggle/input/datasets/nunenuh/pytorch-challange-flower-dataset/cat_to_name.json') as f:
    cat_to_name = json.load(f)

print(cat_to_name)

{'21': 'fire lily', '3': 'canterbury bells', '45': 'bolero deep blue', '1': 'pink primrose', '34': 'mexican aster', '27': 'prince of wales feathers', '7': 'moon orchid', '16': 'globe-flower', '25': 'grape hyacinth', '26': 'corn poppy', '79': 'toad lily', '39': 'siam tulip', '24': 'red ginger', '67': 'spring crocus', '35': 'alpine sea holly', '32': 'garden phlox', '10': 'globe thistle', '6': 'tiger lily', '93': 'ball moss', '33': 'love in the mist', '9': 'monkshood', '102': 'blackberry lily', '14': 'spear thistle', '19': 'balloon flower', '100': 'blanket flower', '13': 'king protea', '49': 'oxeye daisy', '15': 'yellow iris', '61': 'cautleya spicata', '31': 'carnation', '64': 'silverbush', '68': 'bearded iris', '63': 'black-eyed susan', '69': 'windflower', '62': 'japanese anemone', '20': 'giant white arum lily', '38': 'great masterwort', '4': 'sweet pea', '86': 'tree mallow', '101': 'trumpet creeper', '42': 'daffodil', '22': 'pincushion flower', '2': 'hard-leaved pocket orchid', '54': 's

# demak barcha gullar eng kami 33 rasmi bor ekan

In [ ]:
label_counts = df['label'].value_counts()
rare = label_counts[label_counts < 33]
print("33 tadan kam rasmli gullar:")
print(rare)
print("\nJami:", len(rare), "ta gul")

33 tadan kam rasmli gullar:
Series([], Name: count, dtype: int64)

Jami: 0 ta gul


In [ ]:
print(label_counts.sort_values())

label
7      34
1      35
34     35
26     36
6      36
     ... 
73    166
89    169
46    175
77    226
51    234
Name: count, Length: 102, dtype: int64


In [ ]:
cat_to_uzbek = {
    '1': 'pushti primula', '2': 'qattiq bargli cho\'l gulaabi', '3': 'qo\'ng\'iroqchalar',
    '4': 'ingliz qo\'ng\'irog\'i', '5': 'ingliz sariguli', '6': 'yo\'lbars zanbag\'i',
    '7': 'oy orkide', '8': 'jannat qushi', '9': 'bo\'ridoshgul', '10': 'sharsimon xasan',
    '11': 'snaqdargon', '12': 'buzoq oyog\'i', '13': 'qirol proteya', '14': 'shar gul',
    '15': 'sariq iris', '16': 'sharsimon gul', '17': 'purpur konuscha', '18': 'peruan zambag\'i',
    '19': 'balon guli', '20': 'ulkan oq arum zambag\'i', '21': 'olov zambag\'i',
    '22': 'yostiqcha guli', '23': 'fritillariya', '24': 'qizil zanjabil', '25': 'uzum giatsint',
    '26': 'makkajo\'xori ko\'knori', '27': 'uels shahzodasi patlari', '28': 'ulkan moychechak',
    '29': 'artishot', '30': 'shirin no\'xat', '31': 'qalam gul', '32': 'bog\' floksi',
    '33': 'tuman ichidagi sevgi', '34': 'meksika astri', '35': 'alp dengiz badani',
    '36': 'lab guli', '37': 'kaap guli', '38': 'ulkan shamama', '39': 'siam lolasi',
    '40': 'limon daliyas', '41': 'barbeton romashkasi', '42': 'nargis', '43': 'qilich zambag\'i',
    '44': 'qizil zanjabil', '45': 'chuqur ko\'k bolero', '46': 'devor guli', '47': 'sarigul',
    '48': 'sariq sarg\'ich', '49': 'ko\'z romashkasi', '50': 'umumiy qoqio\'t',
    '51': 'petuniya', '52': 'yovvoyi binafsha', '53': 'primula', '54': 'quyosh guli',
    '55': 'pelargoniya', '56': 'llandaff yepiskopi', '57': 'gaura', '58': 'geraniya',
    '59': 'to\'q sariq dahlia', '60': 'sariq-pushti gul', '61': 'kautleya spikata',
    '62': 'yapon anemon', '63': 'qora ko\'zli suzan', '64': 'kumushbuta',
    '65': 'kaliforniya ko\'knori', '66': 'osteospermum', '67': 'bahor krokusi',
    '68': 'soqolli iris', '69': 'shamolcha', '70': 'daraxt ko\'knori', '71': 'gazaniya',
    '72': 'azaleya', '73': 'suv zambag\'i', '74': 'atirgul', '75': 'tikan olma',
    '76': 'ertalabki shon', '77': 'ishtiyoq guli', '78': 'lotos', '79': 'qo\'g\'a zambag\'i',
    '80': 'anthurium', '81': 'frangipani', '82': 'klematis', '83': 'gibiskus',
    '84': 'kolumbina', '85': 'cho\'l atirguli', '86': 'daraxt xamoli', '87': 'magnoliya',
    '88': 'siklamen', '89': 'suv jajzig\'i', '90': 'kanna zambag\'i', '91': 'gippeastrum',
    '92': 'asalari balsami', '93': 'qoqio\'t', '94': 'tulki qo\'lqopchasi', '95': 'bougainvillea',
    '96': 'kameliya', '97': 'saksovulgul', '98': 'meksika petuniyasi', '99': 'bromeliya',
    '100': 'adyol guli', '101': 'trompet o\'rmalagich', '102': 'ezgil'
}

print("O'zbekcha nomlar tayyorlandi!")
print("Jami:", len(cat_to_uzbek), "ta gul")

O'zbekcha nomlar tayyorlandi!
Jami: 102 ta gul


In [ ]:
for k in sorted(cat_to_uzbek.keys(), key=lambda x: int(x)):
    print(f"{k}: {cat_to_uzbek[k]}")

1: pushti primula
2: qattiq bargli cho'l gulaabi
3: qo'ng'iroqchalar
4: ingliz qo'ng'irog'i
5: ingliz sariguli
6: yo'lbars zanbag'i
7: oy orkide
8: jannat qushi
9: bo'ridoshgul
10: sharsimon xasan
11: snaqdargon
12: buzoq oyog'i
13: qirol proteya
14: shar gul
15: sariq iris
16: sharsimon gul
17: purpur konuscha
18: peruan zambag'i
19: balon guli
20: ulkan oq arum zambag'i
21: olov zambag'i
22: yostiqcha guli
23: fritillariya
24: qizil zanjabil
25: uzum giatsint
26: makkajo'xori ko'knori
27: uels shahzodasi patlari
28: ulkan moychechak
29: artishot
30: shirin no'xat
31: qalam gul
32: bog' floksi
33: tuman ichidagi sevgi
34: meksika astri
35: alp dengiz badani
36: lab guli
37: kaap guli
38: ulkan shamama
39: siam lolasi
40: limon daliyas
41: barbeton romashkasi
42: nargis
43: qilich zambag'i
44: qizil zanjabil
45: chuqur ko'k bolero
46: devor guli
47: sarigul
48: sariq sarg'ich
49: ko'z romashkasi
50: umumiy qoqio't
51: petuniya
52: yovvoyi binafsha
53: primula
54: quyosh guli
55: pelarg

# * o'zbekcha nom bilan atalgan ustun qo'shildi

In [ ]:
df['label_uz'] = df['label'].map(cat_to_uzbek)

In [ ]:
df

,path,label,split,label_uz
0,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train,oy orkide
1,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train,oy orkide
2,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train,oy orkide
3,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train,oy orkide
4,/kaggle/input/datasets/nunenuh/pytorch-challan...,7,train,oy orkide
...,...,...,...,...
7365,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid,qoqio't
7366,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid,qoqio't
7367,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid,qoqio't
7368,/kaggle/input/datasets/nunenuh/pytorch-challan...,93,valid,qoqio't


# data qatorlari aralashtirildi

In [ ]:
df = df.sample(frac=1).reset_index(drop=True)
print("Aralashtirildi!")
print(df.head())

Aralashtirildi!
                                                path label  split  \
0  /kaggle/input/datasets/nunenuh/pytorch-challan...    81  train   
1  /kaggle/input/datasets/nunenuh/pytorch-challan...    51  train   
2  /kaggle/input/datasets/nunenuh/pytorch-challan...    89  train   
3  /kaggle/input/datasets/nunenuh/pytorch-challan...    41  train   
4  /kaggle/input/datasets/nunenuh/pytorch-challan...    58  train   

              label_uz  
0           frangipani  
1             petuniya  
2         suv jajzig'i  
3  barbeton romashkasi  
4             geraniya  


In [ ]:
df.shape

(7370, 4)

In [ ]:
print(df['label_uz'].value_counts())

label_uz
petuniya             234
ishtiyoq guli        226
devor guli           175
suv jajzig'i         169
suv zambag'i         166
                    ... 
siam lolasi           36
yo'lbars zanbag'i     36
pushti primula        35
meksika astri         35
oy orkide             34
Name: count, Length: 101, dtype: int64



# train_transform → "train rasmlarga shu amallarni qil" degan qoida

# val_transform   → "valid rasmlarga faqat o'lchamni o'zgartir" degan qoida

# Shunda har bir rasm o'qilganda avtomatik qo'llaniladi.


In [ ]:
from torchvision import transforms

# Train uchun augmentatsiya (ko'proq o'zgartirish)
train_transform = transforms.Compose([ # Bu funksiya bir nechta amallarni bitta ro'yxatga jamlaydi.
    transforms.Resize((224, 224)), # Har qanday o'lchamdagi rasmni (masalan, 500x300 yoki 1000x800) aniq $224 \times 224$ piksel o'lchamiga keltiradi
    transforms.RandomHorizontalFlip(),
# Rasmni tasodifiy ravishda gorizontal (chapdan-o'ngga) aylantiradi. Bu modelga gulning "orqa tomoni" yoki "teskari tomoni" ham o'sha gul ekanligini o'rgatadi.
    transforms.RandomRotation(30), # Rasmni 30 gradusgacha tasodifiy burchakka buradi.
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
# Yorug'lik va kontrastni o'zgartiradi. Bu modelga yorug' yoki qorong'i muhitdagi gullarni ham adashmay topishga yordam beradi.
    transforms.ToTensor(), # Rasm (PILLOW formatidagi) ni matematik tensorga (0 dan 1 gacha bo'lgan sonlar massiviga) o'tkazadi.
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5])
])
# Piksel qiymatlarini ma'lum bir oraliqqa (o'rtacha qiymat 0 bo'lishi uchun) keltiradi. Bu modelning tezroq o'rganishiga (konvergensiyaga) yordam beradi.

# Valid uchun oddiy (o'zgartirmasdan)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5])
])

print("Transformlar tayyor!")

Transformlar tayyor!


### 70 tadan kam rasmli gullar topildi

#         ↓

### Har biridan yangi rasm yasaldi

### (burish, teskari qilish, rang o'zgartirish)

 #        ↓

### Yangi rasmlar /kaggle/working/augmented/ ga saqlandi

  #       ↓

## df ga qo'shildi

## Augmentatsiya yo'li bilan rasmi kam gullar 70 tagacha ko'paytirildi

In [ ]:
import shutil
from PIL import Image, ImageOps, ImageEnhance
import random
import os

aug_dir = '/kaggle/working/augmented'
os.makedirs(aug_dir, exist_ok=True)

def augment_image(img):
    ops = [
        # ops ro'yxati: Bu yerda rasm ustida bajarilishi mumkin bo'lgan 4 xil "o'zgarish" funksiyasi yozilgan.
        lambda x: x.transpose(Image.FLIP_LEFT_RIGHT), # Chap-o'ngga o'girish
        lambda x: x.rotate(random.randint(-30, 30)), # 30 gradusgacha aylantirish
        lambda x: ImageEnhance.Brightness(x).enhance(random.uniform(0.7, 1.3)), # Yorqinlik
        lambda x: ImageEnhance.Contrast(x).enhance(random.uniform(0.7, 1.3)), # Kontrast
    ]
    return random.choice(ops)(img)
 # Har safar chaqirilganda, shu 4 ta amaldan bittasini tasodifiy tanlab, rasmga qo'llaydi. Bu orqali bitta rasmdan yangi versiyalar hosil qilinadi.

MIN = 70
new_rows = []

for label, group in df[df['split'] == 'train'].groupby('label'): # Datasetni har bir gul turi bo'yicha guruhlaydi.
    count = len(group)
    if count < MIN:
        needed = MIN - count # Nechta rasm yetishmayotganini hisoblaydi.
        samples = group.sample(needed, replace=True).reset_index(drop=True)

        for i, row in samples.iterrows():
            img = Image.open(row['path']).convert('RGB')
            img = augment_image(img)
            save_path = f"{aug_dir}/{label}_{i}_aug.jpg"
            img.save(save_path)
            new_rows.append({
                'path': save_path,
                'label': label,
                'split': 'train',
                'label_uz': row['label_uz']
            })

    # Bu qism olingan rasmni augment_image funksiyasidan o'tkazadi, ya'ni uni biroz o'zgartiradi va
# yangi fayl sifatida /kaggle/working/augmented papkasiga saqlaydi.

# new_rows.append(...): Yangi yaratilgan rasmlarning yoli va labeli haqidagi malumotni saqlab qoyadi.
        print(f"{row['label_uz']}: {count} → {MIN} ✅")

aug_df = pd.DataFrame(new_rows)
df = pd.concat([df, aug_df]).reset_index(drop=True)
print(f"\nJami yangi rasm: {len(aug_df)}")
print(f"Umumiy dataset: {len(df)}")
# pd.concat: Eski df (asl rasmlar) bilan yangi aug_df (yaratilgan sun'iy rasmlar) ni bir-biriga yopishtirib, bitta katta jadval hosil qiladi.

pushti primula: 27 → 70 ✅
sharsimon xasan: 38 → 70 ✅
adyol guli: 35 → 70 ✅
trompet o'rmalagich: 49 → 70 ✅
ezgil: 36 → 70 ✅
snaqdargon: 68 → 70 ✅
qirol proteya: 38 → 70 ✅
shar gul: 44 → 70 ✅
sariq iris: 38 → 70 ✅
sharsimon gul: 36 → 70 ✅
purpur konuscha: 60 → 70 ✅
peruan zambag'i: 65 → 70 ✅
balon guli: 38 → 70 ✅
qattiq bargli cho'l gulaabi: 49 → 70 ✅
ulkan oq arum zambag'i: 46 → 70 ✅
olov zambag'i: 34 → 70 ✅
yostiqcha guli: 47 → 70 ✅
qizil zanjabil: 35 → 70 ✅
uzum giatsint: 34 → 70 ✅
makkajo'xori ko'knori: 33 → 70 ✅
uels shahzodasi patlari: 36 → 70 ✅
ulkan moychechak: 55 → 70 ✅
artishot: 62 → 70 ✅
qo'ng'iroqchalar: 36 → 70 ✅
shirin no'xat: 61 → 70 ✅
qalam gul: 48 → 70 ✅
bog' floksi: 36 → 70 ✅
tuman ichidagi sevgi: 31 → 70 ✅
meksika astri: 28 → 70 ✅
alp dengiz badani: 33 → 70 ✅
lab guli: 62 → 70 ✅
ulkan shamama: 44 → 70 ✅
siam lolasi: 33 → 70 ✅
ingliz qo'ng'irog'i: 44 → 70 ✅
limon daliyas: 54 → 70 ✅
nargis: 49 → 70 ✅
chuqur ko'k bolero: 33 → 70 ✅
sarigul: 61 → 70 ✅
sariq sarg'ich: 57 → 7

In [ ]:
df.shape

(9060, 4)

## train va valid ga ajratildi

In [ ]:
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'valid'].reset_index(drop=True)

print("Train:", len(train_df))
print("Valid:", len(val_df))

# Bizning dastlabki kodimizda (for split in ['train', 'valid', 'test']:) fayllar allaqachon papkalar bo‘yicha ajratilgan edi. Ya'ni:
# Dataset papkasini ichida train/ papkasi bor edi.
# valid/ papkasi bor edi.
# test/ papkasi bor edi.

Train: 8242
Valid: 818


# # dataloader qismi yuqorida malumotlarni tayyorlash yakunlandi

In [ ]:
from torch.utils.data import DataLoader

train_dataset = FlowerDataset(train_df, transform=train_transform)
val_dataset   = FlowerDataset(val_df,   transform=val_transform)
# Bu yerda biz oldinroq yaratgan FlowerDataset klassidan ikkita "obyekt" hosil qilyapmiz.
# train_df ga train_transform (augmentatsiyalar bilan) ulanmoqda.
# val_df ga val_transform (faqat resize va to-tensor bilan) ulanmoqda.

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
# batch_size=32: Model barcha rasmlarni bittada o'qimaydi. U bir safarda 32 ta rasm oladi, ularni tahlil qiladi, xatolarini to'g'irlaydi va keyin navbatdagi 32 ta rasmga o'tadi. Bu xotirani tejash uchun juda muhim.
# shuffle=True (Train uchun): O'qitish paytida rasmlar har safar har xil tartibda kelishi kerak. Agar rasmlar har doim bir xil ketma-ketlikda kelsa, model "yodlab" olishi mumkin. Shuffle buni oldini oladi.
# shuffle=False (Validatsiya uchun): Imtihon paytida (validatsiya) tartib muhim emas, shuning uchun uni o'zgartirish shart emas.

print("DataLoader tayyorlandi!")
print("Turlar soni:", len(train_dataset.classes))

DataLoader tayyorlandi!
Turlar soni: 102


## Bunda biz dastlab FlowerCNN dan foydalangan edik yaxshi natija bermagani uchun pastda avvaldan o'qitilga efficientnet dan foydalandik yani fine tuning qildik

In [ ]:
from torchvision import models
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Bu kompyuterimizda GPU (Video karta) borligini tekshiradi. Agar bo'lsa, modelni unga yuklaydi. GPU bo'lmasa, sekinroq bo'lsa ham CPU (protsessor) dan foydalanadi.

model = models.efficientnet_b0(pretrained=True) # modelni noldan o'rgatmayabmiz. U allaqachon "tajribali" holatda keladi. U rasmlarni tanish bo'yicha millionlab soatlik "tajriba"ga ega.
model.classifier[1] = __import__('torch.nn', fromlist=['nn']).Linear(1280, len(train_dataset.classes))
# EfficientNet aslida 1000 xil narsani (masalan, mushuk, it, mashina, samolyot) ajratish uchun o'qitilgan.
# Lekin bizga 102 turdagi gul kerak. Shuning uchun biz modelning oxirgi "bosh qismi"ni (classifier) olib
# tashlaymiz va o'rniga aynan 102 ta klassga moslangan yangi qatlam qo'yamiz. Bu jarayon Fine-tuning deyiladi.

import torch.nn as nn
model.classifier[1] = nn.Linear(1280, len(train_dataset.classes))
model = model.to(device)

print("✅ EfficientNet tayyor!")
print("Device:", device)
print("Turlar soni:", len(train_dataset.classes))

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 130MB/s] 


✅ EfficientNet tayyor!
Device: cuda
Turlar soni: 102


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss() # Model xato qilganda, bu funksiya xatoning kattaligini hisoblab beradi.
optimizer = optim.Adam(model.parameters(), lr=0.001)
# Bu modelning "tuzatuvchisi". U xatolarni kamaytirish uchun modeldagi milliardlab parametrlarni (weightlarni) bir oz o‘zgartirib boradi. lr=0.001
best_acc = 0.0

for epoch in range(10):
    # --- TRAIN ---
    model.train()
    total_loss, correct = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad() # Oldingi qadamdan qolgan "eski" hisob-kitoblarni tozalaydi.
        outputs = model(images) # Model rasmlarni ko‘rib, "bu atirgul", "bu binafsha" deb taxmin qiladi.
        loss = criterion(outputs, labels) # : Modelning taxminini haqiqiy javob bilan solishtiradi.
        loss.backward() # Model qayerda xato qilganini orqaga qarab (backpropagation) aniqlaydi.
        optimizer.step() # Aniqlangan xatolar asosida model parametrlarini yangilaydi (o‘rganadi).
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    train_acc = correct / len(train_dataset) * 100

    # --- VALID ---
    model.eval() # Modelga "o‘rganishni to‘xtat, endi faqat sinab ko‘r" deymiz (ba'zi qatlamlar - masalan, Dropout - ishlamay qoladi).
    val_correct = 0
    with torch.no_grad(): # Imtihon paytida model hech narsani o‘rganmasligi kerak, shuning uchun "xotira" (gradientlar) hisoblanmaydi. Bu kompyuterni juda tezlashtiradi.
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
    val_acc = val_correct / len(val_dataset) * 100

    print(f"Epoch {epoch+1}/10 | Loss: {total_loss:.1f} | Train: {train_acc:.1f}% | Valid: {val_acc:.1f}%")

    # Eng yaxshi modelni saqlash
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f"  💾 Saqlandi! Best: {best_acc:.1f}%")

Epoch 1/10 | Loss: 298.5 | Train: 73.7% | Valid: 91.6%
  💾 Saqlandi! Best: 91.6%
Epoch 2/10 | Loss: 70.9 | Train: 92.4% | Valid: 93.6%
  💾 Saqlandi! Best: 93.6%
Epoch 3/10 | Loss: 42.6 | Train: 95.1% | Valid: 92.7%
Epoch 4/10 | Loss: 40.2 | Train: 95.6% | Valid: 93.0%
Epoch 5/10 | Loss: 32.9 | Train: 96.4% | Valid: 95.2%
  💾 Saqlandi! Best: 95.2%
Epoch 6/10 | Loss: 27.2 | Train: 96.9% | Valid: 93.8%
Epoch 7/10 | Loss: 27.6 | Train: 96.8% | Valid: 94.0%
Epoch 8/10 | Loss: 27.6 | Train: 96.7% | Valid: 94.9%
Epoch 9/10 | Loss: 23.6 | Train: 97.3% | Valid: 96.0%
  💾 Saqlandi! Best: 96.0%
Epoch 10/10 | Loss: 25.0 | Train: 97.1% | Valid: 95.2%


## Gradio interface

## AI qo'shildi aniqlangan gul haqida malumotlar berish uchun

In [ ]:
# Anthropic kutubxonasini

# !pip install anthropic -q
!pip install groq -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:0000:01


In [ ]:
!pip install groq -q

from groq import Groq

groq_client = Groq(api_key="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")

def get_flower_info(flower_name): # Gul nomini oladi → Groq AI ga so'rov yuboradi → o'zbekcha ma'lumot qaytaradi.
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": f"{flower_name} guli haqida o'zbekcha qisqacha ma'lumot ber: kelib chiqishi, xususiyatlari, qayerda o'sadi, qiziqarli faktlar. 3-4 jumlada."
            } # AI ga savol matnini yuboradi. f"..." — ichiga o'zgaruvchi qo'yish imkonini beradi.
        ]
    )
    return response.choices[0].message.content # AI ning javobini qaytaradi.

def predict_and_info(image):
    img = val_transform(image).unsqueeze(0).to(device) # Rasmni modelga mos formatga o'tkazadi. unsqueeze(0) — batch o'lchami qo'shadi.
    with torch.no_grad(): # Gradientni hisoblash o'chiriladi — tezroq ishlaydi, xotira tejaladi.
        output = model(img) # Rasm modelga beriladi, natija chiqadi — 102 ta gul uchun sonlar.
        probs = torch.softmax(output, dim=1) # Sonlarni foizga o'tkazadi — hammasi qo'shilsa 100% bo'ladi.
        confidence = probs.max().item()
        pred = output.argmax(1).item() # Eng yuqori sonning indeksini oladi — bu gul raqami.

    label_num = train_dataset.classes[pred]
    uzbek_name = cat_to_uzbek.get(label_num, label_num)
    # Label raqamini o'zbekcha nomga o'giradi. Topilmasa raqamning o'zini qaytaradi.

    if confidence < 0.5:
        gul_nomi = "⚠️ Aniqlab bo'lmadi"
        info = "Rasmni yaqinroqdan oling yoki boshqa rasm yuklang."
    else:
        gul_nomi = f"🌸 Bu — {uzbek_name}! ({confidence*100:.0f}%)"
        info = get_flower_info(uzbek_name) # Groq AI dan gul haqida ma'lumot oladi.

    return gul_nomi, info

demo = gr.Interface( # Gradio UI yaratadi. fn — qaysi funksiyani ishlatishini ko'rsatadi.
    fn=predict_and_info,
    inputs=gr.Image(type="pil"), # Kirish — rasm yuklash maydoni.
    outputs=[
        gr.Text(label="Gul nomi"),
        gr.Text(label="📖 Batafsil ma'lumot")
    ],
    title="🌸 Gul Tanish",
    description="Gul rasmini yuklang!"
)

demo.launch(share=True)
# Gradio UI ni ishga tushiradi. share=True — internet orqali ochiq havola yaratadi. 🌸

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://6d47f51f9e19172d66.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

In [ ]:
for k in sorted(cat_to_uzbek.keys(), key=lambda x: int(x)):
    print(f"{k}: {cat_to_uzbek[k]}")

1: pushti primula
2: qattiq bargli cho'l gulaabi
3: qo'ng'iroqchalar
4: ingliz qo'ng'irog'i
5: ingliz sariguli
6: yo'lbars zanbag'i
7: oy orkide
8: jannat qushi
9: bo'ridoshgul
10: sharsimon xasan
11: snaqdargon
12: buzoq oyog'i
13: qirol proteya
14: shar gul
15: sariq iris
16: sharsimon gul
17: purpur konuscha
18: peruan zambag'i
19: balon guli
20: ulkan oq arum zambag'i
21: olov zambag'i
22: yostiqcha guli
23: fritillariya
24: qizil zanjabil
25: uzum giatsint
26: makkajo'xori ko'knori
27: uels shahzodasi patlari
28: ulkan moychechak
29: artishot
30: shirin no'xat
31: qalam gul
32: bog' floksi
33: tuman ichidagi sevgi
34: meksika astri
35: alp dengiz badani
36: lab guli
37: kaap guli
38: ulkan shamama
39: siam lolasi
40: limon daliyas
41: barbeton romashkasi
42: nargis
43: qilich zambag'i
44: qizil zanjabil
45: chuqur ko'k bolero
46: devor guli
47: sarigul
48: sariq sarg'ich
49: ko'z romashkasi
50: umumiy qoqio't
51: petuniya
52: yovvoyi binafsha
53: primula
54: quyosh guli
55: pelarg

In [ ]:
print("Model joyi: /kaggle/working/best_model.pth")

Model joyi: /kaggle/working/best_model.pth


In [ ]:
import shutil
shutil.copy('/kaggle/working/best_model.pth', '/kaggle/working/best_model_download.pth')
print("✅ Tayyor!")

✅ Tayyor!


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error